# Script 1

#### Objective
The next notebook takes info from CVs, transform this info to it's vector embedding representation, feeds this data into a database and allows to search in the database the CVs that suits the most to a query.

### 1. Initializing a local client

Qdrant db client can work in the cloud or locally in three ways: In-memory, in hard drive and in a docker container (Prefered for production-ready projects). Here "In-memory" mode will be used for testing purposes.

To make a quick test for the embedding model, a simple embedding model is used the the in-memory feature is used for Qdrant. ":memory:" Allows Qdrant to work in in-memory mode, so the database lives in my PC's RAM.  One can work with the database as long as the notebook kernel is not restarted, or the code snippet down below is not re-ran

In [4]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")

### 2. Creating collections

A qdrant collection is the fundamental piece of information of the vectorial database. It's an isolated container containing the vectors, IDs and metadata (data points). It's the equivalent of a Table in a SQL database.

In [38]:
models_to_test = {
    "GIST-all-MiniLM-L6-v2": {"size": 384, "model_name": "avsolatorio/GIST-all-MiniLM-L6-v2"},
    # # Models down below are for script 2
    # "bge-base-en-v1.5": {"size": 768, "model_name": "BAAI/bge-base-en-v1.5"},
    # "harrier-oss-v1-0.6b": {"size": 1024, "model_name": "microsoft/harrier-oss-v1-0.6b"},
    # "Qwen3-Embedding-4B": {"size": 2560, "model_name": "Qwen/Qwen3-Embedding-4B"}
}


# Create a unique collection for each model
for name, config in models_to_test.items():
    COLLECTION_NAME = f"test_{name}"
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=config["size"],
            distance=models.Distance.COSINE
        )
    )
print("Collections created successfully!")

ValueError: Collection test_GIST-all-MiniLM-L6-v2 already exists

### 3. Loading CV data and embedding the data

Qdrant cloud has something called "Cloud Inference" in which the data is converted to embeddings with a fixed model. However, for Qdrant local, we have to do that process ourselves (Unless we use fastembed), but we are free to use any model, which is exactly what I want for this and script 2. The thing with fastembed is that for models that are not default, additional configurations must be done, which is not ideal for this case, so I'll just use Huggin Face's sentence-transformers

In [14]:
import json

with open("../data/cv_extracted_info.json", "r") as file:
    cv_data = json.load(file)

print(cv_data)

[{'name': 'Cristian Benjamin García Casierra', 'email': 'ing_cristian_garcia@outlook.com', 'phone': '+57 3160546495', 'skills': ['CSS', 'JAVA', 'EAGLE', 'C++', 'JAVASCRIPT', 'DART', 'PYTHON', 'HTML', '.NET', 'TYPESCRIPT', 'NODE.JS', 'TRABAJO EN EQUIPO', 'APPIUM', 'SELENIUM IDE', 'FLUTTER', 'REACT', 'ANGULAR', 'UX/UI', 'ENGLISH', 'SPANISH'], 'experience': ['Desarrollador front end en AFO IMPROVEMENT, Cali (abr. 2020 - mar. 2025): Diseño e implementación de interfaces de usuario con tecnologías HTML, CSS, JavaScript y React. Integración con servicios backend, mediante el consumo de APIs RESTful y GraphQL. Optimización del rendimiento y experiencia del usuario (UX). Diseño y ejecución de pruebas automatizadas en Selenium para aplicaciones web y en Appium para aplicaciones móviles. Implementación de scripts en Java.', 'Ingeniero de Proyectos en PIXEL INGENIERÍA S.A.S, Santiago De Cali (nov. 2018 - dic. 2021): Diseño y desarrollo de placas de circuito impreso. Gestión de programas de diseño

Loading the vector embedding model

In [ ]:
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

# Load environment variables from the .env file in the root directory
load_dotenv("../.env")
HUGGING_FACE_API_KEY = os.getenv("HUGGING_FACE_API_KEY")

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("avsolatorio/GIST-all-MiniLM-L6-v2", token=HUGGING_FACE_API_KEY)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8227.57it/s]


Serializing the cv profiles and counting the number of tokens specific to the model

In [34]:
serialized_cvs = []
for idx, cv in enumerate(cv_data):
    
    # 1. Manually serialize the dictionary into human-readable text
    serialized_cv = (
        f"name: {cv['name']}. "
        f"email: {cv['email']}. "
        f"skills: {cv['skills']}"
        f"phone: {cv['phone']}. "
        f"experience: {cv['experience']}. "
        f"education: {cv['education']}. "
    )
    num_tokens = len(model.tokenizer.encode(serialized_cv))
    max_tokens = model.max_seq_length
    if num_tokens > max_tokens:
        print(f"serialized CV for {cv['name']} has {num_tokens} tokens, which is greater than the number of tokens this model supports ({max_tokens}).")
    serialized_cvs.append(serialized_cv)

serialized CV for Cristian Benjamin García Casierra has 643, which is greater than the number of tokens this model supports (512)


Vector embedding the CVs

In [36]:
embeddings = model.encode(serialized_cvs).tolist()
print(f"embeddings.shape: {embeddings.shape}")

embeddings.shape: (3, 384)


### 4. Inserting the data to the database

In [45]:
EMBEDDING_MODEL = models_to_test["GIST-all-MiniLM-L6-v2"]['model_name']

client.upload_points(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=idx,
            vector=embeddings[idx],
            # Store the original dictionary (to keep queryable fields) and the serialized text (useful for LLM context later)
            payload={
                **cv,
                "serialized_text": serialized_cvs[idx]
            }
        )
        for idx, cv in enumerate(cv_data)
    ],
)

### 5. Querying to the database

In [48]:
hits = client.query_points(
    collection_name=COLLECTION_NAME,
    query=model.encode("experience with Tableau").tolist(),
    limit=3,
).points

hits

[ScoredPoint(id=2, version=0, score=0.7782818776235325, payload={'name': 'Yajat S Murthy', 'email': 'yajatsmurthy@gmail.com', 'phone': '+61 0490 517 598', 'skills': ['AWS Sagemaker', 'Azure Open AI', 'Langchain', 'RAG', 'Pytorch', 'Tensorflow', 'Pandas', 'Keras', 'Scikit Learn', 'MongoDB', 'RedShift', 'Dynamo DB', 'MySQL', 'Postgres', 'Pine Cone', 'Weaviate', 'Python', 'JavaScript', 'TypeScript', 'C++', 'Java', 'Solidity', 'Shell Scripting', 'Dart', 'Bash', 'R', 'AWS Lambda', 'SQS', 'EKS', 'EMR', 'React Native', 'Expo', 'Nest JS', 'Apache Spark', 'Dbt', 'Kafka', 'Microservice architecture', 'REST API', 'Git', 'Linux', 'Prisma', 'NestJS', 'PySpark', 'AWS', 'Docker', 'Kubernetes', 'Terraform', 'Jenkins'], 'experience': ['ML Developer, Ansrsource India, Bengaluru (February 2024 - January 2025)', 'Software Engineering Intern, Skia Coffee Opc Pvt. Ltd (November 2023 - January 2024)', 'Graduate Internship, Central Food Research Institute, India (October 2022 - April 2023)'], 'education': ['M